In [3]:
import asyncio
import time

from langchain.chat_models import init_chat_model

from langchain_demo.config import load_project_environment, require_environment_variable

load_project_environment(override=True)
api_key = require_environment_variable("DEEPSEEK_API_KEY")
base_url = require_environment_variable("DEEPSEEK_API_BASE")

model = init_chat_model(model="deepseek:deepseek-flash", api_key=api_key, base_url=base_url)

### 异步请求
* 避免阻塞主线程
* 优化资源

#### ainvoke

In [ ]:
async def ask(question: str):
    start_time = time.time()
    await asyncio.sleep(1)
    answer = await model.ainvoke(question)
    print(f"{question}: {answer.content},\nAsk  time: {time.time() - start_time}")


async def ask_async():
    start_time = time.time()
    task1 = asyncio.create_task(ask("翻译成英文：春天来了"))
    task2 = asyncio.create_task(ask("翻译成英文：夏天走了"))
    task3 = asyncio.create_task(ask("翻译成英文：秋天凉了"))

    await asyncio.gather(task1, task2, task3)
    loop_time = time.time() - start_time
    print(f"all time: {loop_time}")


# jupyter中不需要main方法，直接调用就可以
await ask_async()

### astream
* 生产中很少使用这种多个流使用同一个连接的情况,通常每个个请求都会有各自独立的连接

In [ ]:
async def ask_stream(task_name: str, question: str) -> None:
    start_time = time.perf_counter()

    print(f"[{task_name}] 开始请求")

    try:
        async for chunk in model.astream(question):
            if chunk.content:
                # chunk前增加任务表示，方便看出流式输出结果（多个请求的chunk可能会出现在同一行）
                print(
                    f"[{task_name}] {chunk.content}",
                    end="",
                    flush=True,
                )
        elapsed = time.perf_counter() - start_time
        print(f"\n[{task_name}] 完成，耗时：{elapsed:.2f}s")

    except Exception as exc:
        elapsed = time.perf_counter() - start_time
        print(f"\n[{task_name}] 请求失败，耗时：{elapsed:.2f}s，错误：{type(exc).__name__}: {exc}")
        raise


async def ask_async() -> None:
    start_time = time.perf_counter()

    await asyncio.gather(
        ask_stream("春天", "翻译成英文，并增加意境描述：春天来了"),
        ask_stream("夏天", "翻译成英文，并增加意境描述：夏天走了"),
        ask_stream("秋天", "翻译成英文，并增加意境描述：秋天凉了"),
    )

    elapsed = time.perf_counter() - start_time
    print(f"\n全部完成，总耗时：{elapsed:.2f}s")


await ask_async()

### abatch

In [5]:
async def ask_batch(messages: list[str]):
    start_time = time.perf_counter()
    for response in await model.abatch(messages):
        print(response.content)
    print(f'ask time :{time.perf_counter() - start_time}')

async def do_batch():
    start_time = time.perf_counter()
    messages1 = [
        "翻译成英文：春天来了",
        "翻译成英文：夏天很热",
    ]
    messages2 = [
        "翻译成英文：秋天凉了",
        "翻译成英文：冬天冷了",
    ]
    task1 = asyncio.create_task(ask_batch(messages1))
    task2 = asyncio.create_task(ask_batch(messages2))
    await asyncio.gather(task1, task2)
    print(f'total time :{time.perf_counter() - start_time}')

await do_batch()

Spring has arrived.
Summer is very hot.
ask time :1.6806737000588328
Autumn has turned cool.
Winter is cold.
ask time :1.7306013000197709
total time :1.7307879000436515
